# An?lise explorat?ria

Este notebook carrega amostras de loans e s?ries macro (FRED) e cria gr?ficos e tabelas iniciais.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw'
STAGING_DIR = DATA_DIR / 'staging'


## Carregar amostra de loans


In [ ]:
loan_files = list(RAW_DIR.rglob('*.csv'))
print('Arquivos raw:', len(loan_files))
loans = None
if loan_files:
    loans = pd.read_csv(loan_files[0])
    display(loans.head())


## Carregar s?ries macro (FRED)


In [ ]:
fred_files = [p for p in RAW_DIR.rglob('*.csv') if 'fred' in p.name.lower()]
macro = None
if fred_files:
    macro = pd.read_csv(fred_files[0])
    display(macro.head())


## S?rie: pre?o m?dio


In [ ]:
if macro is not None and 'date' in macro.columns and 'value' in macro.columns:
    macro['date'] = pd.to_datetime(macro['date'], errors='coerce')
    macro['value'] = pd.to_numeric(macro['value'], errors='coerce')
    plt.figure(figsize=(10,4))
    sns.lineplot(data=macro, x='date', y='value')
    plt.title('S?rie macro (ex.: pre?o m?dio)')
    plt.show()


## Taxa de default ao longo do tempo


In [ ]:
if loans is not None and 'date' in loans.columns and 'delinquency' in loans.columns:
    loans['date'] = pd.to_datetime(loans['date'], errors='coerce')
    loans['delinquency'] = pd.to_numeric(loans['delinquency'], errors='coerce')
    tmp = loans.dropna(subset=['date']).groupby(pd.Grouper(key='date', freq='M'))['delinquency'].mean()
    tmp = tmp.reset_index()
    plt.figure(figsize=(10,4))
    sns.lineplot(data=tmp, x='date', y='delinquency')
    plt.title('Taxa m?dia de delinquency ao longo do tempo')
    plt.show()


## Tabelas resumidas (delinquency por vintage)


In [ ]:
if loans is not None and 'vintage' in loans.columns and 'delinquency' in loans.columns:
    tbl = loans.groupby('vintage')['delinquency'].mean().reset_index()
    display(tbl.head())
